## Deep Learning Assignment 2 CS60010 Spring Semester 2025
### Part C: Building a BERT-based Classifier for Model Identification

In this notebook we use the BERT Model to classify captions generated under different occlusion levels into two classes:
- CustomCaptioningModel
- smolVLM

Inputs given are of the format
` <original_caption> <SEP> <generated_caption> <SEP> <occlusion_percentage> `

We then evaluate the model using the scores : precision, recall, f1-score

In [1]:
from transformers import BertTokenizer, BertModel
import torch.nn as nn

2025-04-13 16:16:34.463685: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744560994.488081      89 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744560994.495401      89 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


### Preparing Data

In [ ]:
from torch.utils.data import Dataset

class TextPairDataset(Dataset):
    """
    A Dataset for handling text classification data.
    
    This dataset prepares text inputs and their corresponding labels for model training,
    handling the tokenization process for each text input.

    Methods:
        __init__:
            Initialize the dataset with input texts and labels.
            
            Args:
                dataframe: Pandas DataFrame containing 'input_text' and 'label' columns
                tokenizer: Pre-trained tokenizer for encoding the text

        __len__:
            Return the total number of samples in the dataset.

        __getitem__:
            Get a single data sample at the specified index.
        
            Args:
                idx: Index of the sample to fetch
                
            Returns:
                dict: Dictionary containing:
                    - input_ids: Tokenized and encoded input text
                    - attention_mask: Mask indicating which tokens should be attended to
                    - label: Corresponding label for the input text
                    - input_text: Original input text 
    """
    def __init__(self, dataframe, tokenizer):
        self.tokenizer = tokenizer
        self.inputs = dataframe['input_text'].tolist()
        self.labels = dataframe['label'].tolist()

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.inputs[idx],
            padding='max_length',
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(self.labels[idx]),
            'input_text': self.inputs[idx], 
        }

### CaptionClassifier
This implementation uses the BERT model for text classification tasks. The architecture consists of:

- Pre-trained BERT Base Model from Hugging Face
- Classification Head: A simple but effective classifier built on top of BERT's output representations.

The model utilizes the special `[CLS]` token output from BERT, which is designed to capture the semantic information of the entire input sequence. This representation is then passed through the classification head to determine the final prediction.

In [ ]:
class CaptionClassifier(nn.Module):
    """
    A neural network model for text classification using BERT.
    
    Methods:
        __init__:
            Initialize the classifier with a pre-trained BERT model and classification layers.

        forward:
            Forward pass through the model.
        
            Args:
                input_ids: Tensor of token ids from the tokenizer
                attention_mask: Tensor indicating which tokens should be attended to
                
            Returns:
                logits: Un-normalised prediction scores for each class
    """
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2) 
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # Use [CLS] token
        logits = self.classifier(cls_output)
        return logits

In [4]:
from sklearn.model_selection import train_test_split
import pandas as pd
import re

def extract_number(filename):
    return int(re.search(r'test_(\d+)', filename).group(1))

custom_paths = ['/kaggle/input/captions-10apr/generated_captions.csv', '/kaggle/input/captions-10apr/captions-custom-10.csv', 
               '/kaggle/input/captions-10apr/captions-custom-50.csv', '/kaggle/input/captions-10apr/captions-custom-80.csv']

smolvlm_paths = ['/kaggle/input/captions-10apr/hypotheses_occlusion_0.csv', '/kaggle/input/captions-10apr/hypotheses_occlusion_10.csv', 
                '/kaggle/input/captions-10apr/hypotheses_occlusion_50.csv', '/kaggle/input/captions-10apr/hypotheses_occlusion_80.csv']

occlusion_levels = [0, 10, 50, 80]
combined_data = []

for i in range(4):
    
    original_df = pd.read_csv('/kaggle/input/captions-10apr/test.csv')  
    custom_df = pd.read_csv(custom_paths[i])  
    smolvlm_df = pd.read_csv(smolvlm_paths[i])  

    # print("original_df type:", type(original_df))
    # print("original_df shape:", original_df.shape)
    # print("original_df.columns:", original_df.columns.tolist())
    # print("original_df.head():\n", original_df.head())
    
    original_df.columns = ['sno', 'filename', 'caption']
    custom_df.columns = ['filename', 'caption']
    smolvlm_df.columns = ['filename', 'caption']

    custom_df['file_id'] = custom_df['filename'].apply(extract_number)
    smolvlm_df['file_id'] = smolvlm_df['filename'].apply(extract_number)
    original_df['file_id'] = original_df['filename'].apply(extract_number)

    custom_df = custom_df.sort_values('file_id').reset_index(drop=True)
    smolvlm_df = smolvlm_df.sort_values('file_id').reset_index(drop=True)
    original_df = original_df.sort_values('file_id').reset_index(drop=True)

    perturbation = occlusion_levels[i]

    for idx in range(len(custom_df)):
        file_id = custom_df.loc[idx, 'file_id']
        original_caption = original_df.loc[idx, 'caption']
        custom_caption = custom_df.loc[idx, 'caption']
        smolvlm_caption = smolvlm_df.loc[idx, 'caption']
    
        combined_data.append({
            'input_text': f"{original_caption} <SEP> {custom_caption} <SEP> {perturbation}",
            'label': 1,
            'file_id': file_id
        })
        combined_data.append({
            'input_text': f"{original_caption} <SEP> {smolvlm_caption} <SEP> {perturbation}",
            'label': 0,
            'file_id': file_id
        })

dataset_df = pd.DataFrame(combined_data)

unique_ids = dataset_df['file_id'].unique()
train_ids, test_ids = train_test_split(unique_ids, test_size=0.15, random_state=42)

train_df = dataset_df[dataset_df['file_id'].isin(train_ids)].reset_index(drop=True)
test_df = dataset_df[dataset_df['file_id'].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)} samples")
print(f"Test: {len(test_df)} samples")


Train: 6304 samples
Test: 1120 samples


In [5]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_dataset = TextPairDataset(train_df, tokenizer)
test_dataset = TextPairDataset(test_df, tokenizer)

from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

### Training the classifier

In [ ]:
def train_classifier(model, dataloader, optimizer, criterion, device, epochs=3):
    """
    Train a text classification model.
    
    Args:
        model: The model to train
        dataloader: DataLoader containing the training data batches
        optimizer: PyTorch optimizer for parameter updates
        criterion: Loss function (CrossEntropyLoss)
        device: Device to run on (cuda/cpu)
        epochs: Number of epochs
    """
    model.to(device)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in dataloader:
            optimizer.zero_grad()

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

### Evaluating the Model

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, classification_report
import torch

def evaluate_classifier(model, dataloader, device):
    """
    Evaluate the classification model.

    Parameters:
        model
        dataloader: Test data loader.
        device: Device to use ('cuda' or 'cpu').

    Returns:
        dict: Precision, Recall, and F1 scores (macro-averaged).
    """
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro'
    )

    print("Macro Precision:", precision)
    print("Macro Recall:", recall)
    print("Macro F1 Score:", f1)

    print("\nFull classification report:")
    print(classification_report(all_labels, all_preds, target_names=["SmolVLM", "Custom"]))

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1
    }

In [8]:
import torch
from torch import nn, optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CaptionClassifier().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)

train_classifier(
    model=model,
    dataloader=train_loader,
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    epochs=3
)

metrics = evaluate_classifier(
    model=model,
    dataloader=test_loader,
    device=device
)

print("\nEvaluation Metrics:", metrics)


Epoch 1/3, Loss: 0.1769
Epoch 2/3, Loss: 0.0602
Epoch 3/3, Loss: 0.0469
Macro Precision: 0.9731195160203276
Macro Recall: 0.9723214285714286
Macro F1 Score: 0.9723097511625997

Full classification report:
              precision    recall  f1-score   support

     SmolVLM       0.99      0.95      0.97       560
      Custom       0.95      0.99      0.97       560

    accuracy                           0.97      1120
   macro avg       0.97      0.97      0.97      1120
weighted avg       0.97      0.97      0.97      1120


Evaluation Metrics: {'precision': 0.9731195160203276, 'recall': 0.9723214285714286, 'f1_score': 0.9723097511625997}


In [12]:
def save_predictions(model, dataloader, device, output_filename="predictions.csv"):
    model.eval()
    predictions = []
    input_texts = []
    true_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'] 
            texts = batch['input_text']

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs, dim=1)

            predictions.extend(preds.cpu().tolist())
            input_texts.extend(texts)
            true_labels.extend(labels.cpu().tolist())

    df = pd.DataFrame({
        'input_text': input_texts,
        'true_label': true_labels,
        'predicted_label': predictions
    })

    df.to_csv(output_filename, index=False)
    print(f" Predictions saved to {output_filename}")


In [10]:
# print(len(test_loader.dataset))

In [13]:
save_predictions(model, test_loader, device)

 Predictions saved to predictions.csv
